# Statistical Analysis of Hospital Patient Outcomes

## Introduction

This notebook uses inferential statistical methods to evaluate whether patient characteristics and admission-related factors are associated with hospital length of stay and in-hospital mortality.

The analysis examines relationships involving patient age, gender, admission type, insurance coverage, length of stay, and mortality status. Pearson correlation, Welch’s t-test, one-way ANOVA, and chi-square tests are applied based on the type of variables being analyzed.

A significance level of 0.05 is used for all hypothesis tests. Results with a p-value below 0.05 are considered statistically significant, while results above 0.05 indicate insufficient evidence to reject the null hypothesis.

The objective is to determine which factors demonstrate statistically supported relationships with hospital utilization and patient outcomes, rather than relying only on apparent differences in the data.
This notebook evaluates relationships between patient demographics, admission characteristics, hospital length of stay, and mortality using inferential statistical methods. The results support findings presented in the accompanying Power BI dashboard.

In [ ]:
%pip install scipy

In [ ]:
import pandas as pd
from scipy import stats
from scipy.stats import ttest_ind
from scipy.stats import f_oneway
from scipy.stats import chi2_contingency


In [ ]:
df = pd.read_csv('patient_analysis_view.csv')

### Pearson Correlation Test
### Purpose:
 Determine whether patient age is linearly related to hospital length of stay.

 Null Hypothesis (H0):
 There is no linear relationship between age and length of stay.

 Alternative Hypothesis (H1):
 There is a linear relationship between age and length of stay.

In [6]:
r_coefficient, p_value = stats.pearsonr(df['anchor_age'], df['length_of_stay'])

print(f"Pearson Correlation Coefficient: {r_coefficient:.4f}")
print(f"P-value (Statistical Significance): {p_value:.4f}")

# Interpretation
if p_value < 0.05:
    print("Result: Reject H0.")
    print("There is a statistically significant relationship between age and length of stay.")
else:
    print("Result: Fail to reject H0.")
    print("No statistically significant linear relationship exists between age and length of stay.")

print(f"Correlation Strength: {r_coefficient:.4f}")

Pearson Correlation Coefficient: -0.0277
P-value (Statistical Significance): 0.6478
Result: Fail to reject H0.
No statistically significant linear relationship exists between age and length of stay.
Correlation Strength: -0.0277


### T-test compares two group LOS
 
H0: Average length of stay is the same for males and females.\
H1: Average length of stay differs between males and females.

In [14]:
male_los = df[df['gender'] == 'M']['length_of_stay']
female_los = df[df['gender'] == 'F']['length_of_stay']

t_stat, p_value = ttest_ind(
    male_los,
    female_los,
    equal_var=False   
)
print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")

if p_value < 0.05:
    print("Result: Reject H0.")
    print("Average hospital stay differs significantly between males and females.")
else:
    print("Result: Fail to reject H0.")
    print("No significant difference in average hospital stay between males and females.")

T-statistic: 1.5260044358419975
P-value: 0.12817815158760798
Result: Fail to reject H0.
No significant difference in average hospital stay between males and females.


### One-Way ANOVA Determines whether at least one admission type has a different average length of stay.

H0:
All admission types have the same average LOS.

H1:
At least one admission type has a different average LOS.

In [ ]:
groups = [
    group['length_of_stay'].values
    for _, group in df.groupby('admission_type')
]

f_stat, p_value = f_oneway(*groups)

print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")


if p_value < 0.05:
    print("Result: Reject H0.")
    print("Length of stay differs among admission types.")
else:
    print("Result: Fail to reject H0.")
    print("No significant difference in length of stay among admission types.")

F-statistic: 6.4263
P-value: 0.0000
Result: Reject H0.
Length of stay differs among admission types.


### Chi-Square Test of Independence Determines whether gender and hospital mortality are associated.

H0:
Gender and mortality are independent.

H1:
Gender and mortality are associated.

In [ ]:

observed = pd.crosstab(
    df['gender'],
    df['hospital_expire_flag']
)

print(observed)

chi2, p_value, dof, expected = chi2_contingency(observed)

print(f"Chi-square: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Reject H0.")
    print("Mortality is associated with patient gender.")
else:
    print("Result: Fail to reject H0.")
    print("No evidence that mortality depends on gender.")

hospital_expire_flag    0   1
gender                       
F                     130   3
M                     130  12
Chi-square: 3.9802
P-value: 0.0460
Result: Reject H0.
Mortality is associated with patient gender.


### Tests whether mortality rates differ across admission types.

In [10]:
observed = pd.crosstab(
    df['admission_type'],
    df['hospital_expire_flag']
)

chi2, p_value, dof, expected = chi2_contingency(observed)

print(f"Chi-square: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Reject H0.")
    print("Mortality is associated with admission type.")
else:
    print("Result: Fail to reject H0.")
    print("No significant relationship between admission type and mortality.")

Chi-square: 8.7751
P-value: 0.3616
Result: Fail to reject H0.
No significant relationship between admission type and mortality.


### Create binary variable:
1 = patient stayed at least 7 days
0 = patient stayed fewer than 7 days

In [11]:
df['long_stay'] = (df['length_of_stay'] >= 7).astype(int)

observed = pd.crosstab(
    df['long_stay'],
    df['hospital_expire_flag']
)

chi2, p_value, dof, expected = chi2_contingency(observed)

print(f"Chi-square: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Reject H0.")
    print("Long hospital stays are associated with mortality.")
else:
    print("Result: Fail to reject H0.")
    print("No significant relationship between long stays and mortality.")

Chi-square: 1.9509
P-value: 0.1625
Result: Fail to reject H0.
No significant relationship between long stays and mortality.


### Tests whether average length of stay differs by insurance provider.

In [12]:
groups = [
    group['length_of_stay'].values
    for _, group in df.groupby('insurance')
]

f_stat, p_value = f_oneway(*groups)

print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Reject H0.")
    print("Average length of stay differs across insurance types.")
else:
    print("Result: Fail to reject H0.")
    print("No significant difference in average length of stay across insurance types.")

F-statistic: 0.5951
P-value: 0.5522
Result: Fail to reject H0.
No significant difference in average length of stay across insurance types.


# Key Findings

- Pearson correlation analysis found no statistically significant linear relationship between patient age and hospital length of stay, suggesting age alone is not a strong predictor of hospitalization duration.
- Welch's t-test found no statistically significant difference in average length of stay between male and female patients.
- One-way ANOVA showed that admission type significantly affects hospital length of stay, indicating certain admission pathways are associated with longer hospitalizations.
- Chi-square analysis was used to evaluate whether patient gender was associated with in-hospital mortality. The results should be interpreted according to the reported p-value.
- No statistically significant association was found between admission type and mortality if the p-value exceeded the significance level.
- Patients with hospital stays of seven days or longer were not significantly more likely to experience in-hospital mortality if the chi-square test was not significant.
- One-way ANOVA found no statistically significant differences in average length of stay among insurance providers, suggesting that insurance type alone does not explain variation in hospitalization duration.

## Conclusion

Overall, the analyses indicate that most patient demographic characteristics examined were not independently associated with substantial differences in hospital outcomes. Admission type was the strongest factor influencing length of stay, while age, gender, and insurance type showed limited or no statistically significant effects. These findings suggest that additional clinical variables—such as diagnosis, disease severity, and comorbidities—would likely improve the explanation of patient outcomes.